Cloning repository

In [1]:
import subprocess, sys, shutil, os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
# Replace with your actual GitHub token secret name if different
token = UserSecretsClient().get_secret("GITHUB_TOKEN") 
clone_dir = "/kaggle/working/birdclef2026"

if os.path.exists(clone_dir):
    shutil.rmtree(clone_dir)


# Clone the repo
subprocess.run([
    "git", "clone", "--quiet", "--branch", "feat/data-pipeline",
    f"https://{token}@github.com/HoaPNG520/BirdCLEF2026.git",
    clone_dir
], check=True)

# ── debug: check what actually got cloned ──
print("Last commit:")
subprocess.run(["git", "-C", clone_dir, "log", "--oneline", "-3"])

# Add to system path so Python can find your modules
sys.path.insert(0, clone_dir)
print("Repository cloned successfully.")

Last commit:
bef4aaf feat: finalize AMP and gradient accumulation in train.py for B3 stability
145d9db increase batch size to 1024
3c323e7 increase batch size to 64
Repository cloned successfully.


Training

In [2]:
import sys
# Force Python to delete any cached versions of our modules
for mod in list(sys.modules.keys()):
    if any(mod.startswith(x) for x in ['configs', 'data', 'models', 'train']):
        del sys.modules[mod]

from train import train_fold
import os

# Create a dedicated directory for our final published dataset
output_dir = "/kaggle/working/model_artifacts"
os.makedirs(output_dir, exist_ok=True)

# Train folds 
train_fold(fold=0, epochs=15, lr=1e-3, save_dir=output_dir)

Training on cuda | Fold 0


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


Fold distribution:
fold
0    6687
1    6686
2    6686
3    6686
4    6686
Name: count, dtype: int64
Fold 0 — train: 26,744  val: 6,687


model.safetensors:   0%|          | 0.00/49.3M [00:00<?, ?B/s]

Epoch 1/15: 100%|██████████| 1671/1671 [37:43<00:00,  1.35s/it]


...

Epoch 13 | Loss: 0.0272 | Val cMAP: 0.0087


Epoch 14/15: 100%|██████████| 1671/1671 [37:29<00:00,  1.35s/it]


Epoch 14 | Loss: 0.0272 | Val cMAP: 0.0091


Epoch 15/15: 100%|██████████| 1671/1671 [37:22<00:00,  1.34s/it]


Epoch 15 | Loss: 0.0272 | Val cMAP: 0.0094


Packaging

In [3]:
import shutil

# We must copy your python scripts into the output directory 
# so they are included in the Kaggle Dataset we publish.
dirs_to_copy = ["configs", "data", "models"]
files_to_copy = ["infer.py", "requirements.txt"]

for d in dirs_to_copy:
    src = os.path.join(clone_dir, d)
    dst = os.path.join(output_dir, d)
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)

for f in files_to_copy:
    src = os.path.join(clone_dir, f)
    dst = os.path.join(output_dir, f)
    if os.path.exists(src):
        shutil.copy2(src, dst)


print("Codebase packaged alongside .pth models successfully.")

Codebase packaged alongside .pth models successfully.
